# Robot Demo 4: camera tracking with a ROT3U arm (v0.3)

Copyright (c) 2026 Tico Hannan. BSD-3-Clause.

The camera on the gripper is the only sensor. The notebook finds the block by color, measures how the arm
moves the picture, and steers the base and the wrist with a PID per joint so the block stays centered.

**Before you start** (first time: follow `docs/GETTING-STARTED.md` in this repository, step by step):
1. Your robot has a `robots/<your robot>/robot_config.h` with its joints mapped by `servo_tester`
   (Sir Pinch-a-lot's is in `robots/sir_pinch_a_lot_revB/`).
2. `firmware/servo_bridge/servo_bridge.ino` (v0.3) is uploaded to the Mega and the **Arduino Serial Monitor is
   closed** (only one program can use the port). Servo power on. The arm can stay limp until part B.

**How to read this notebook.** Every section starts with a box like this one, saying what the cells do and
whether to run them:

| Label | Meaning |
| --- | --- |
| **REQUIRED** | run it every session, in order |
| **ONCE** | needed once on this laptop (install) or once per object and room (colors) |
| **OPTIONAL** | a check or a measurement; skip it when things work |
| **SKIP IF ...** | the condition under which you can leave it out |

| Part | Sections | Result |
| --- | --- | --- |
| 0 | Settings, imports | everything adjustable in one cell |
| A | Color picker, object profile, camera diagnostics, camera, color check | a camera that works and a color range for this object and room |
| B | Connect, start pose (pictures), power up | the arm holding the start pose |
| C | Noise and image Jacobian, camera geometry, slop, ringing and input shaping, step response | the numbers the tracker needs |
| D | Track (1 axis, 2 axes), compare detectors, finish, save the session | the demo video and logs |

With `PORT = "SIM"` everything runs on a simulator (no Arduino, no camera); the color picker section is skipped.

## 0.1 Settings

**What it does:** Every value you might change: port, object and room, defaults, start pose, display, tracking gains, smoothing, where to aim, and the camera mount on the gripper.

**Run:** **REQUIRED**, every session, first. Edit, then run. Run it again after any change (then re-run the cells that use the value).

`PROFILE = None` is where the **paste block** goes: section A.3 prints a few lines starting with
`PROFILE = dict(` for this object and room. Paste them over the `PROFILE = None` line, and later sessions start
from them. Without a paste block the notebook looks for `profiles/<object>__<room>.json`, and without that it
uses the defaults below.

In [ ]:
# ===================== 0.1 Settings (REQUIRED: edit, then run) =====================
PORT = "COM3"                        # the Arduino's port, or "SIM" for the simulator (no arm, no camera)
ROBOT_CONFIG = "../robots/sir_pinch_a_lot_revB/robot_config.h"   # YOUR robot's file (docs/GETTING-STARTED.md, part 4)

# ---- what is tracked, and where (the lighting): together they name the object profile
OBJECT = "green wooden block #1"
ROOM = "living room"
PROFILE = None                       # <- paste the block printed by section A.3 over this line
BLOCK_MM = 24.8                      # side of the block in mm (measure yours): used for the distance estimate

# ---- defaults, used when no profile exists for OBJECT and ROOM
DEFAULT_CAMERA = dict(index=1, name="HBVCAM camera", width=640, height=480, fps=30, fmt=None,
                      orientation=[0, True, True])   # [rotate cw deg, flip horizontal, flip vertical]
DEFAULT_COLOR = None                 # "green", "red", "yellow" or "wood"; None = guess from OBJECT's name
CAPTURE_DIR = "captures"             # where the color picker saves captures
PROFILE_DIR = "profiles"             # where object profiles are saved
CAMERA_SIZE = (640, 480)             # tracking size: 640 x 480 gave about 9 frames/s on the HBVCAM (1280 x 720: 7.5)

# ---- arm and display
POSE = "WALL"                        # "WALL": camera looks straight ahead; "TABLE": camera looks down at the table
ROLL_DEG = 0.0                       # wrist roll in the poses. 0 puts the gripper servo (and the camera) on top on
                                     # Sir Pinch-a-lot; if yours is underneath at 0, use 180 (GETTING-STARTED.md part 5)
SHOW = "window"                      # "window" (OpenCV window, best for screen recording), "inline", or None
METHOD = "blob"                      # "blob" (class method) or "edges" (color-gated edges)

# ---- tracking
TRACK_V, TRACK_A = 500, 2500         # sketch speed (us/s) and acceleration (us/s^2) limits while tracking. Lower = smoother
KP, KI, KD = (0.3, 0.3), (3.0, 3.0), (0.02, 0.02)   # PID gains per joint (base, wrist)
DEADBAND = None                      # (in_px, out_px), or None = suggested by the measurements in part C
SHAPING = "measure"                  # input shaping against the frame's swing: "measure" (use section C.4's fit),
                                     # (f_hz, zeta) to set it by hand, or None for off
SHAPING_DEFAULT = (2.3, 0.3)         # used with "measure" if C.4 was skipped (2026-11-15 step response: about 2.3 Hz)
AIM = "camera"                       # "camera": block to the image center; "gripper": block onto the gripper's z axis
OVERLAY = True                       # draw the camera center, gripper z axis and x5/y5 arrows on the tracking view

# ---- camera on the gripper, DH frame 5, mm: YOUR camera's numbers (docs/GETTING-STARTED.md part 5.3).
#      The values below are Sir Pinch-a-lot's (HBVCAM on gripper_camera_bracket rev 2.1).
MOUNT = dict(pos_5=(29.5, 0.0, -55.0),   # optical center: 29.5 mm toward +x5 (above the gripper), 55 mm back from the grasp point
             up_5=(1.0, 0.0, 0.0),       # image up = +x5
             right_5=(0.0, 1.0, 0.0),    # image right = +y5
             f_px=720.0, ref_width=640)  # focal length in pixels at 640 wide (estimate; section C.2 checks it)

## 0.2 Imports, robot and session

**What it does:** Loads the Python package and your robot's `robot_config.h` (pulse calibration, pulse limits, gripper, link lengths), closes anything left open by an earlier run of this notebook, and makes the helper `make_rig()`, which every measurement cell calls so it always uses the camera and link that are open now.

**Run:** **REQUIRED**, after 0.1.

**You should see:** `rot3u_demo4 0.3.0`, your robot's name, one line per joint with its pulse formula and limits in µs and degrees, the session stamp, and SIM or HARDWARE.

**If it fails:** `ModuleNotFoundError`: open the notebook from the `demo4` folder of the repository (the one with `rot3u_demo4` in it). A `ValueError` about `robot_config.h` says what is missing in that file; `FileNotFoundError`: check `ROBOT_CONFIG` in 0.1.

In [ ]:
# ===================== 0.2 Imports and session (REQUIRED) =====================
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import rot3u_demo4
from rot3u_demo4 import link, robot, vision, control, sim, camdiag, profiles, geometry, robot_view

for _name in ("lk", "cam", "picker"):                 # running this cell again releases the port and the camera
    _o = globals().get(_name)
    if _o is not None:
        try: _o.close()
        except Exception: pass
        globals()[_name] = None
SIM = (PORT == "SIM")
STAMP = time.strftime("%Y%m%d_%H%M%S")
for d in ("logs", "videos", CAPTURE_DIR, PROFILE_DIR):
    os.makedirs(d, exist_ok=True)
mount = geometry.CameraMount(**MOUNT)
robot.load_config(ROBOT_CONFIG)
for _p in robot.POSES.values():
    _p[0][4] = ROLL_DEG
print(robot.describe())
SESSION = {"stamp": STAMP, "object": OBJECT, "room": ROOM, "pose": POSE, "robot": robot.ROBOT_NAME}   # filled in as you go; saved in D.5

def make_rig():
    """A rig from what is open now (link, camera, color range, method)."""
    return control.make_rig(lk, cam, COLOR_RANGE, METHOD)

print("rot3u_demo4", rot3u_demo4.__version__, "| session", STAMP, "|", "SIMULATOR" if SIM else f"HARDWARE on {PORT}")

# Part A: camera and colors

The color range comes from the **color picker** (`camera-color-picker`): it shows the live camera, lets you
set up the camera (which one, size, flip/rotate, exposure), and samples the block's color in a circle.
Several captures of the same block, at different angles and places in the picture, give a color range that
holds up while the arm moves. They are packed into an **object profile** (object + room + camera settings + color).

If a profile for this object and room already exists (pasted in 0.1 or saved in `profiles/`), skip to **A.4**.

## A.1 Color picker installed?

**What it does:** Checks that the color picker package can be imported, and says how to install it if not.

**Run:** **ONCE** per laptop (the check itself is quick: run it whenever unsure). **SKIP IF** you are not taking new captures.

**You should see:** `camera_color_picker` and its version.

**If it fails:** Uncomment the `%pip install` line, run the cell, then restart the kernel (Kernel > Restart) and run 0.1 and 0.2 again.

In [ ]:
# ===================== A.1 Color picker installed? (ONCE) =====================
# %pip install git+https://github.com/ticohannan/camera-color-picker
try:
    import camera_color_picker
    print("camera_color_picker", getattr(camera_color_picker, "__version__", "?"), "is installed")
except ImportError:
    print("not installed: uncomment the %pip line above, run this cell, then restart the kernel")

## A.2 Take color captures with the picker

**What it does:** Opens the color picker on the gripper camera. You set up the camera there, then save a series of **shotgun** captures of the same block at different angles and positions.

**Run:** **SKIP IF** a profile for this OBJECT and ROOM already exists and the colors still look right in A.6. Otherwise **ONCE** per object and room (and again if the lighting changes a lot).

**You should see:** the picker panel with the live picture.

**If it fails:** black or striped picture: try another mode in the picker, or run A.5 (diagnostics) first. 'camera busy': close the Windows Camera app, Teams, Zoom, and stop the camera in section A.4 (`cam.close()`).

**Camera set-up, in the picker:**
1. **source:** the gripper camera (*HBVCAM camera*), then **Start**.
2. **mode:** 640 x 480 (the size the tracker uses; it is faster than 1280 x 720), driver-default format. **Apply**.
3. **rotate / flip** until the picture is upright as the camera sees the world (on this bracket: flip horizontal **and** flip vertical).
4. **Camera settings:** if the dialog allows it, set exposure and white balance to manual so the colors do not drift. **Read camera properties** saves them with each capture.

**Sampling set-up:**
5. **sample: Cluster**, **pattern: random (shotgun)**, **points: 300**, **exclude outliers: OFF** (outliers are part of the real spread).
6. **diameter:** hold the block about as far from the camera as during tracking (25 to 35 cm). Set the diameter so the circle
   covers **as much of the block's face as possible without touching a finger, the edge of the block, or the background**
   (about 3/4 of the block's width in the picture). Hold the block by its back or bottom so your fingers stay out of the circle.

**The series (6 to 10 captures of the same block):** **Capture still** after each change:
7. face on, in the image center;
8. turned about 30 deg left, then right; tilted up, then down (the faces catch the light differently);
9. at the left and right edges and the top and bottom of the picture (the lens darkens the corners);
10. nearer (20 cm) and farther (40 cm);
11. with the room's lights as they will be during the demo; one capture with a hand's shadow on the block if shadows happen.

Before each capture, look at the circle: only block inside. If one slipped, uncheck it in A.3.
12. When done, **stop the picker** with the next cell: only one program can use the camera.

In [ ]:
# ===================== A.2 Color picker (SKIP IF a profile exists) =====================
if SIM:
    print("simulator: no camera to sample; skipped")
else:
    from camera_color_picker import ColorPicker
    PICKER_STARTED = time.time()
    picker = ColorPicker(save_dir=CAPTURE_DIR, layout="side").show()

In [ ]:
# ---- A.2 stop the picker (run after the last capture: it releases the camera)
if globals().get("picker") is not None:
    picker.close(); picker = None
    print("picker stopped, camera released")
else:
    print("no picker running")

## A.3 Build the object profile from the captures

**What it does:** Shows the captures saved since the picker was started (or the newest 12) as thumbnails with check boxes. Uncheck any whose circle took in a finger, the background or a glare spot. The next cell pools the checked captures into a profile (mean and standard deviation of H, S, V), shows the color range on each capture's own picture, saves the profile, and prints the **paste block**.

**Run:** **SKIP IF** you skipped A.2. Otherwise run both cells, then **paste the printed block into 0.1** over `PROFILE = None`.

**You should see:** in the second cell: the profile line (H, S, V mean ± sigma and the range), each capture with the in-range pixels tinted magenta, and the paste block.

**If it fails:** warnings about another camera or orientation: those captures were taken with other settings; uncheck them. Large 'in range' percentages (over about 15%) mean background is in range: uncheck the capture that pulls the range wide.

In [ ]:
# ===================== A.3a choose captures (SKIP IF A.2 was skipped) =====================
selector = None
if SIM:
    print("simulator: skipped")
else:
    _since = globals().get("PICKER_STARTED")
    _files = profiles.captures_since(CAPTURE_DIR, _since - 5 if _since else None)
    if not _since:
        _files = _files[-12:]
    print(len(_files), "captures in", CAPTURE_DIR, "(since the picker started)" if _since else "(the newest)")
    selector = profiles.CaptureSelector(_files).show() if _files else None

In [ ]:
# ===================== A.3b build, check, save, print the paste block =====================
if selector is None or not selector.selected:
    print("no captures chosen: skipped (the profile comes from 0.1, profiles/ or the defaults)")
else:
    prof_new = profiles.ObjectProfile.from_captures(selector.selected, OBJECT, ROOM, size=CAMERA_SIZE, size_mm=BLOCK_MM)
    print(prof_new)
    grid, frac = profiles.mask_check(prof_new, selector.selected)
    if grid is not None:
        vision.show_inline(grid, width=960)
        print("in range per picture:", ", ".join(f"{100 * f:.1f}%" for f in frac))
    print("saved", prof_new.save(PROFILE_DIR))
    print("\n# ===== copy from here into 0.1, over PROFILE = None =====")
    print(prof_new.paste_block())

## A.4 Use the profile, open the camera

**What it does:** Picks this session's profile (first match: a profile built in A.3 just now, the block pasted in 0.1, the saved file for OBJECT and ROOM, then the defaults), opens the camera with its settings, and shows one frame with a crosshair and 'UP' at the top.

**Run:** **REQUIRED**. Run it again any time to reopen the camera (it closes the old one first).

**You should see:** where the profile came from, the camera line (size, format, frame rate, brightness), and a picture that is **upright** with the crosshair in the middle.

**If it fails:** TimeoutError or a black picture: the picker still has the camera (run the stop cell in A.2), or run A.5. Upside down or mirrored: fix the orientation in the profile (take new captures with the right flips) or set `profile.camera['orientation']` here and re-run.

In [ ]:
# ===================== A.4 Profile and camera (REQUIRED) =====================
if globals().get("prof_new") is not None:
    profile, where = prof_new, "built in A.3 this session"
else:
    profile, where = profiles.resolve(PROFILE, OBJECT, ROOM, PROFILE_DIR)
    if profile is None:
        profile = profiles.default_profile(OBJECT, ROOM, DEFAULT_CAMERA, DEFAULT_COLOR)
        where = "defaults (no captures for this object and room yet: run A.2 and A.3 for a real color range)"
if not profile.size_mm:
    profile.size_mm = BLOCK_MM
mount.block_mm = profile.size_mm
COLOR_RANGE = profile.color_range()
print(f"profile: {profile.tag}  <- {where}")
print(" ", profile.model())
SESSION["profile"] = profile.to_dict()

if globals().get("cam") is not None:
    cam.close(); cam = None
if SIM:
    _hsv = tuple(int(round(v)) for v in profile.model().mean)
    world = sim.SimWorld(realtime=True, pose=POSE, color_hsv=(_hsv[0], max(_hsv[1], 120), max(_hsv[2], 90)),
                         block_mm=profile.size_mm,
                         structure={0: sim.StructureModel(2.4, 0.08), 3: sim.StructureModel(3.2, 0.10)})
    cam = sim.SimCamera(world)
else:
    cam = profile.open_camera()
    time.sleep(2.0)                                    # let exposure settle and the frame-rate counter fill
camdiag.test_card(cam, 560)
print(cam.status())

## A.5 Camera diagnostics

**What it does:** For a fussy camera: (1) lists the cameras the system reports, (2) tries each size and format and measures what the driver really gives (size, format, frames per second, black frames, stripes), and recommends a mode, (3) watches brightness and frame rate for 30 s, (4) compares the driver's settings with the ones saved with the captures. The camera also has a watchdog: 20 black frames in a row, or 3 s without frames, and it reopens itself (`cam.reopens` counts it).

**Run:** **OPTIONAL**. Run when the picture is black, striped, slow, flickering, or the wrong camera. Each cell can be run alone.

**You should see:** a table per cell; the probe takes about 40 s, closes the camera while it tries the modes, then opens it again.

**If it fails:** read the troubleshooting list printed by the last cell.

In [ ]:
# ===================== A.5a which cameras? =====================
_cams = camdiag.cameras() if not SIM else print("simulator: one simulated camera")

In [ ]:
# ===================== A.5b probe sizes and formats (about 40 s; closes the camera) =====================
if not SIM:
    if globals().get("cam") is not None:
        cam.close(); cam = None
    probe_rows = camdiag.probe(profile.camera["index"], orientation=tuple(profile.camera["orientation"]))
    best = camdiag.recommend(probe_rows, min_fps=12)
    print("recommended:", best and f"{best['w']} x {best['h']} {best['fmt_asked'] or 'driver default'} at {best['fps']} frames/s")
    print("to use it: CAMERA_SIZE in 0.1, or profile.camera.update(width=..., height=..., fmt=None), then run A.4")
    cam = profile.open_camera()                       # the probe closed the camera: open it again with the profile
    time.sleep(2.0)
    print(cam.status())
else:
    print("simulator: skipped")

In [ ]:
# ===================== A.5c watch brightness and frame rate for 30 s (arm still, block in view) =====================
watch_rows = camdiag.watch(cam, seconds=30) if cam is not None else print("no camera open: run A.4 first")

In [ ]:
# ===================== A.5d driver settings vs the captures; troubleshooting list =====================
if cam is None:
    print("no camera open: run A.4 first")
elif not SIM:
    camdiag.properties(cam)
    camdiag.compare_properties(cam, profile.camera.get("properties", {}))
    if cam.events:
        print("watchdog:", *[f"{e[1]}" for e in cam.events], sep="\n  ")
print(camdiag.TROUBLESHOOTING)
# cam.settings_dialog()    # Windows camera dialog: exposure, white balance (uncomment to open)
# cam.reopen()             # close and open the camera again with the same settings

## A.6 Color check: both detectors on one frame

**What it does:** Thresholds one frame with the profile's range and runs both detectors: the blob centroid (class method, magenta star) and the color-gated edge rectangle (cyan). Left: the picture; right: the mask.

**Run:** **REQUIRED** (quick). Hold the block where the camera sees it.

**You should see:** the block white in the mask and almost nothing else; both aim points within a few pixels.

**If it fails:** background in the mask: lower k for S or V in A.7 (or take new captures without background). Block missing or patchy: raise k in A.7, or add captures of the missing faces.

In [ ]:
# ===================== A.6 Color check (REQUIRED) =====================
frame, _, _ = cam.read()
mask, blob, edges = vision.detect(frame, COLOR_RANGE, both=True)
vision.show_inline(vision.annotate(frame, mask, blob, edges, lines=[str(COLOR_RANGE)]), width=960)
if blob.found:
    print(f"blob  (class method): ({blob.cx:.1f}, {blob.cy:.1f}), area {blob.area:.0f} px, {blob.n_blobs} blob(s) in range")
    print(f"edges (alternative):  ({edges.cx:.1f}, {edges.cy:.1f}), {edges.size_px[0]:.0f} x {edges.size_px[1]:.0f} px [{edges.note}]")
    print(f"aim points differ by {np.hypot(blob.cx - edges.cx, blob.cy - edges.cy):.1f} px; "
          f"{100 * (mask > 0).mean():.1f}% of the picture is in range")
else:
    print("nothing in the color range: fix the range (A.7) before going on")

## A.7 Tune k live

**What it does:** A live mask with one slider per channel for k (the range is mean ± k·sigma). Then save k into the profile and print a new paste block.

**Run:** **OPTIONAL**. **SKIP IF** A.6 looked clean.

**You should see:** the live mask while you move the sliders.

**If it fails:** nothing updates: run A.4 again (the camera was closed).

In [ ]:
# ===================== A.7a live k sliders (OPTIONAL) =====================
tuner = vision.ThresholdTuner(cam, profile.model(), method=METHOD); tuner.show()

In [ ]:
# ---- A.7b keep the k values: profile saved, new paste block printed
tuner.stop()
profile.set_k(tuner.color.k)
COLOR_RANGE = profile.color_range()
print(profile.model()); print("saved", profile.save(PROFILE_DIR)); print(profile.paste_block())

# Part B: the arm

## B.1 Connect

**What it does:** Opens the serial link to `servo_bridge` (or the simulator) and prints the sketch's greeting and state.

**Run:** **REQUIRED**. Run it again any time: it closes the old connection first.

**You should see:** `HELLO,servo_bridge,0.3,2`, a state line (`moving 0`, active channels as they are: channels stay on if the servo board kept its power through a reset), and 'pulse limits ... sent to the sketch'.

**If it fails:** 'no HELLO': close the Arduino Serial Monitor, check `PORT`, unplug and replug the USB cable. 'refused the limits': the sketch is older than v0.3; upload `firmware/servo_bridge/servo_bridge.ino`.

In [ ]:
# ===================== B.1 Connect (REQUIRED) =====================
if globals().get("lk") is not None:
    lk.close(); lk = None
if SIM:
    lk = link.BridgeLink(sim.SimTransport(world), log_path=f"logs/{STAMP}_link.csv")
else:
    lk = link.BridgeLink(link.SerialTransport(PORT), log_path=f"logs/{STAMP}_link.csv")
hello = lk.connect()
print(hello)
print(lk.get_state())
robot.apply_limits(lk)                               # this robot's pulse limits (the sketch only knows 500-2500 us)
FW_SHAPING = "0.1" not in str(hello)
if not FW_SHAPING:
    print("sketch v0.1: no input shaping. Upload servo_bridge v0.2 for SHAPE.")

## B.2 Start pose: what it should look like

**What it does:** Draws the start pose from the arm's kinematic model and the CAD servo positions: a side view, a top view and a 3D view, and describes it in words. The green square is where the block should be (about 27 cm in front of the camera, at camera height).

**Run:** **REQUIRED** before B.3 the first time; afterwards optional.

**You should see:** for WALL: base facing forward, upper arm straight up, forearm 45 deg up, gripper level and pointing forward, gripper servo and camera on top.

**Set up, before powering up:**
1. Robot on the table, nothing within 40 cm of it. Servo power supply on.
2. **Hold the arm** close to the pose in the pictures (B.3 starts channels that are off one at a time, and an
   unpowered joint jumps to its pulse at full servo speed).
3. Put the block in front of the camera: for WALL, about 25 to 35 cm in front of the lens at about the lens's height
   (the drawing says how high), on a box or taped to the wall. For TABLE, on the table about 15 cm in front of the base.

In [ ]:
# ===================== B.2 Start pose pictures (REQUIRED the first time) =====================
target = robot.pose_us(POSE)
_T = geometry.frames(geometry.q_from_pulses(target))[5] @ mount.T_5c()
_block = _T[:3, 3] + 270.0 * _T[:3, 2]              # 27 cm along the camera's view
if _block[2] < 0:                                   # looking at the table: the block sits on it
    _block = _T[:3, 3] + _T[:3, 2] * (-_T[2, 3] / _T[2, 2]); _block[2] += profile.size_mm / 2
fig = robot_view.show_pose(target, POSE, mount=mount, block=_block); plt.show()
print(robot_view.describe_pose(target, mount))
print(f"block: about x {_block[0]:.0f}, y {_block[1]:.0f}, z {_block[2]:.0f} mm (x forward from the base axis, z up from the base plate)")
print(POSE, "pulses:", target)

## B.3 Power up into the start pose

**What it does:** Starts the channels that are off, base outward, one at a time, then moves everything smoothly to the pose. Channels already holding (servo board stayed powered) are moved smoothly, not jumped.

**Run:** **REQUIRED**. Hold the arm near the pose; type `yes` when asked.

**You should see:** one line per channel, then a state line with `active=63` (all six on) and `moving 0`.

**If it fails:** a channel refused: its pulse is outside the limits (check POSE). The arm sags or buzzes: support it and type `lk.cmd("OFF *")` in a new cell.

In [ ]:
# ===================== B.3 Power up (REQUIRED) =====================
print(POSE, "pulses:", target)
if not SIM and input("Arm supported near the pose, nothing in its way? type yes: ").strip().lower() != "yes":
    raise RuntimeError("not confirmed - nothing was started")
print(robot.power_up(lk, target))
if FW_SHAPING:
    lk.cmd("SHAPE * 0")                             # start with shaping off; C.4 or D.1 turns it on

# Part C: measurements

Put the block where the camera sees it, near the image center, and **leave it still** for every cell in this part.
Each measurement jogs a joint a little and watches the block move in the picture.

## C.1 Noise and the image Jacobian

**What it does:** Measures how much the block's position jitters with the arm still (noise), then jogs the base and the wrist ±40 µs each and measures how far the block moves in the picture: the image Jacobian J (pixels per µs) that turns a pixel error into joint corrections. Also gives lost motion (dead band + backlash) and a suggested dead band.

**Run:** **REQUIRED** once per session and pose (about 30 s).

**You should see:** per joint: px/µs, the direction the block moves, lost motion; the condition number (under 10 is good); a suggested dead band.

**If it fails:** 'not seen steadily': A.6 first (color range, lighting). TimeoutError: the camera stalled; run A.4, then this cell.

In [ ]:
# ===================== C.1 Noise and image Jacobian (REQUIRED) =====================
rig = make_rig()
print(control.measure_noise(rig))
jac = control.measure_jacobian(rig, joints=(0, 3), step_us=40)
print(jac.report())
SESSION["jacobian"] = {"J": jac.J.tolist(), "lost_us": jac.lost_us, "repeat_px": jac.repeat_px, "noise_px": jac.noise_px}

## C.2 Camera geometry check

**What it does:** Estimates the block's distance two ways: from its size in the picture (needs BLOCK_MM and the focal length) and from the base joint's image gain and the camera's distance from the base axis (needs the focal length and the arm model). If they disagree a lot, the focal length is off: the cell prints the focal length that would make them agree. Then shows one frame with the overlay: camera center, the gripper's z axis and where it meets the block's distance (orange circle), x5 (red, image up) and y5 (green, image right), the block's offset from the gripper axis in mm, and a compass with world up and forward.

**Run:** **OPTIONAL** (quick). Recommended once, and whenever the camera or bracket changes.

**You should see:** two distances within about 20% of each other, and the overlay. With the gripper level (WALL), the gripper axis point is below the image center by f·29.5/Z px (about 80 px at 27 cm).

**If it fails:** very different distances: measure the block (BLOCK_MM) and the distance with a ruler, and set `MOUNT['f_px']` to (block size in px) × (distance in mm) / BLOCK_MM.

In [ ]:
# ===================== C.2 Camera geometry (OPTIONAL) =====================
rig = make_rig()
frame, _, _ = cam.read()
_, det = rig.detect(frame)
st = lk.get_state()
W = frame.shape[1]
if det.found:
    Z_size = geometry.depth_from_size(mount, np.sqrt(det.area), W)
    Z_jac = geometry.depth_from_jacobian(jac, st.pos, mount, W)
    print(f"distance from the block's size: {Z_size:.0f} mm (block {mount.block_mm:.0f} mm = {np.sqrt(det.area):.0f} px)")
    print(f"distance from the base gain:     {Z_jac:.0f} mm")
    if np.isfinite(Z_jac) and Z_jac > 0:
        print(f"focal length that makes them agree: about {mount.f_px * Z_jac / Z_size:.0f} px at {mount.ref_width} wide "
              f"(now {mount.f_px:.0f}); a ruler measurement of the distance beats both")
    SESSION["depth_mm"] = {"from_size": Z_size, "from_jacobian": Z_jac, "f_px": mount.f_px}
view, info = geometry.draw_overlay(frame.copy(), mount, det, st.pos)
vision.show_inline(view, width=720)
print({k: (tuple(round(x, 1) for x in v) if isinstance(v, tuple) else (round(v, 1) if v else v)) for k, v in info.items()})

## C.3 Slop staircase

**What it does:** Steps one joint by one PCA9685 count (4.9 µs) at a time, up, down and up again, and watches the block: the loop's width is the slop (dead band + backlash); the biggest single jump shows stick-slip. The suggested dead band is updated with it.

**Run:** **OPTIONAL** (about 40 s per joint). Run when tracking hunts around the center, or to report the slop.

**You should see:** two loop plots (commanded vs seen) and one line per joint; the dead band suggestion.

**If it fails:** 'too close to a limit': move the block so the joint sits nearer the middle of its range, then C.1 again.

In [ ]:
# ===================== C.3 Slop staircase (OPTIONAL) =====================
rig = make_rig()
slop = [control.measure_slop(rig, jac, k) for k in (0, 1)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, s in zip(ax, slop):
    a.plot(s["cmd_us"], s["resp_us"], "o-", ms=3); a.plot(s["cmd_us"], s["cmd_us"], "k:", lw=1)
    a.set_title(f'{robot.JOINT_NAMES[s["joint"]]}: slop {s["width_us"]:.1f} us, biggest jump {s["max_jump"]:.1f} steps')
    a.set_xlabel("commanded change, us"); a.set_ylabel("seen by the camera, us"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print("suggested dead band (in, out) px:", control.suggest_deadband(jac, slop))
SESSION["slop"] = [{k: (float(v) if np.isscalar(v) else None) for k, v in s.items() if np.isscalar(v)} for s in slop]

## C.4 Ringing and input shaping

**What it does:** The frame (brackets, servo horns, the table) springs: after a quick move the camera keeps swinging for a moment even though the servos have stopped. This cell gives the base and then the wrist a quick step, films the swing, and fits a damped sine: its frequency and damping ratio. Then it turns on **input shaping** in the sketch (`SHAPE`): each move is split into two copies, the second delayed by half a swing period, so the second cancels the swing the first starts. Last, it repeats the step with shaping on and reports how much smaller the swing is.

**Run:** **RECOMMENDED** (about 20 s). **SKIP IF** the sketch is v0.1, or set `SHAPING` by hand in 0.1.

**You should see:** per joint: swing in px, frequency (Hz) and damping; with shaping, a much smaller swing (60 to 90% less); plots of both.

**If it fails:** 'too few samples': the swing died out before the camera saw it (nothing to fix), or the frame rate is too low; a frequency near half the frame rate (4.5 Hz at 9 frames/s) cannot be trusted: try a smaller CAMERA_SIZE for more frames/s.

Moves get longer by half a swing period (about 0.2 s at 2.3 Hz): the price of a smooth stop.
The tracker's own gains, its acceleration limit (`TRACK_A`) and the dead band are the other smoothing knobs.

In [ ]:
# ===================== C.4 Ringing and input shaping (RECOMMENDED) =====================
rig = make_rig()
ring = {}
if not FW_SHAPING:
    print("sketch v0.1: no SHAPE command. Upload servo_bridge v0.2.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for k in (0, 1):
        control.apply_shaping(lk, [jac.joints[k]], 0, 0, say=lambda s: None)     # first without shaping
        off = control.measure_ringdown(rig, jac, k, step_us=60)
        if "fit" not in off:
            continue
        f, z = off["fit"]["f_hz"], off["fit"]["zeta"]
        control.apply_shaping(lk, [jac.joints[k]], f, z)
        on = control.measure_ringdown(rig, jac, k, step_us=60)
        ring[jac.joints[k]] = {"f_hz": f, "zeta": z, "pp_px_off": off.get("pp_px"), "pp_px_on": on.get("pp_px")}
        if on.get("pp_us") is not None:
            print(f"  -> {robot.JOINT_NAMES[jac.joints[k]]}: swing {off['pp_px']:.1f} px without shaping, {on['pp_px']:.1f} px with it")
        a = ax[k]
        a.plot(off["t"], off["resp_us"], ".-", label="shaping off"); a.plot(on["t"], on["resp_us"], ".-", label="shaping on")
        a.plot(off["t_fit0"] + off["fit"]["t"], off["fit"]["fitted"], "k:", label=f"fit {f:.2f} Hz, damping {z:.2f}")
        a.set_title(robot.JOINT_NAMES[jac.joints[k]]); a.set_xlabel("s"); a.set_ylabel("block motion, us"); a.legend(fontsize=8); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
SESSION["ringing"] = ring

## C.5 Step response

**What it does:** Commands one 60 µs step on each joint at the tracking speed and acceleration limits and films it: first motion (camera delay + servo start), rise time, overshoot. Use it to choose TRACK_V and TRACK_A.

**Run:** **OPTIONAL** (about 10 s).

**You should see:** two plots (sketch command vs camera) and one line per joint.

**If it fails:** the block hardly moved: the step is inside the servo's dead band; use step_us=100.

In [ ]:
# ===================== C.5 Step response (OPTIONAL) =====================
rig = make_rig()
steps = [control.step_response(rig, jac, k, step_us=60, v=TRACK_V, a=TRACK_A) for k in (0, 1)]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, s in zip(ax, steps):
    a.plot(s["t"], s["sketch_us"], label="sketch (commanded)"); a.plot(s["t"], s["resp_us"], label="camera (real)")
    a.set_title(robot.JOINT_NAMES[s["joint"]]); a.set_xlabel("s"); a.set_ylabel("us"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
SESSION["step"] = [{k: s.get(k) for k in ("joint", "dead_time", "rise_time", "overshoot", "final_us")} for s in steps]

# Part D: tracking

**How the control works.** Each frame, the detector gives the block's position; the error to the aim point (pixels) is
turned into the pulse change each joint still needs with the image Jacobian (`eu = -J^-1 e`). A **PID per joint** turns
that into an absolute pulse target: I does most of the work (it keeps pushing until the servo really moves, which
gets it out of its dead band), P gives a quick first push, D brakes when the error is already shrinking. The integrator
is held back by how far the target is ahead of where the sketch has the servo (anti-windup). Inside the dead band the
targets freeze (CENTERED), so the loop does not hunt around the servo's resolution.

**Smoothing,** from the arm outward: the sketch limits speed and acceleration (`TRACK_V`, `TRACK_A`) and, with v0.2,
shapes each move against the frame's swing (`SHAPE`); the PID's D term and the dead band do the rest.

Keys in the OpenCV window: **q** quit, **1** / **2** one or two axes, **m** switch detector, **space** pause.
With `SHOW = "inline"` or `None`, stop with the notebook's stop button. The arm always ends holding.

## D.1 Tracker settings

**What it does:** Builds the tracker from the settings and the measurements: gains, speed and acceleration, dead band (measured unless set in 0.1), input shaping (measured, by hand, or off), the aim point (image center or gripper axis) and the overlay.

**Run:** **REQUIRED** before D.2 or D.3 (run again after changing a setting).

**You should see:** one summary line.

In [ ]:
# ===================== D.1 Tracker settings (REQUIRED) =====================
db = DEADBAND or control.suggest_deadband(jac, globals().get("slop") or ())
if FW_SHAPING:
    if SHAPING == "measure":
        for j in jac.joints:
            f, z = (ring[j]["f_hz"], ring[j]["zeta"]) if j in globals().get("ring", {}) else SHAPING_DEFAULT
            control.apply_shaping(lk, [j], f, z)
    elif SHAPING:
        control.apply_shaping(lk, list(jac.joints), *SHAPING)
    else:
        control.apply_shaping(lk, list(jac.joints), 0, 0)

def new_tracker(mode):
    cfg = control.TrackerConfig(mode=mode, track_v=TRACK_V, track_a=TRACK_A, kp=KP, ki=KI, kd=KD)
    cfg.deadband_in_px, cfg.deadband_out_px = db
    if AIM == "gripper":
        cfg.aim_fn = geometry.gripper_aim(mount)
    trk = control.Tracker(make_rig(), jac.J, cfg)
    if OVERLAY:
        trk.overlay = geometry.tracker_overlay(mount)
    return trk

def summary(log):
    st = [r["state"] for r in log]
    e = [r["e_px"] for r in log if r["state"] == "CENTERED"]
    out = {s: st.count(s) for s in sorted(set(st))}
    print(out, "| error while CENTERED: mean %.1f px, max %.1f px" % (np.mean(e), np.max(e)) if e else "| never centered")
    return {"states": out, "err_mean_px": float(np.mean(e)) if e else None, "err_max_px": float(np.max(e)) if e else None}

print(f"dead band in {db[0]} px / out {db[1]} px | speed {TRACK_V} us/s, accel {TRACK_A} us/s^2 | "
      f"aim: {AIM} | overlay {'on' if OVERLAY else 'off'} | shaping {SHAPING if FW_SHAPING else 'not available'}")
SESSION["tracker"] = {"deadband_px": list(db), "track_v": TRACK_V, "track_a": TRACK_A, "kp": KP, "ki": KI, "kd": KD,
                      "aim": AIM, "shaping": SHAPING}

## D.2 Track with the base only (1 axis)

**What it does:** Only the base moves: the block is kept on the vertical line through the aim point.

**Run:** **OPTIONAL** (part of the assignment's 1-axis requirement). Press **q** in the window to stop.

**You should see:** state TRACK while the block moves, then CENTERED. A video in `videos/` and a log in `logs/`.

**If it fails:** the window does not open (opencv-python-headless): set `SHOW = "inline"`.

In [ ]:
# ===================== D.2 Track, 1 axis (OPTIONAL) =====================
if SIM: world.target = np.array([450.0, -40.0, 300.0])     # simulator only: move the block
trk1 = new_tracker("1AXIS")
log1 = trk1.run(seconds=8 if SIM else None, show=SHOW, record=f"videos/{STAMP}_1axis.mp4")
trk1.save_log(f"logs/{STAMP}_1axis.csv"); SESSION["track_1axis"] = summary(log1)

## D.3 Track with base and wrist (2 axes)

**What it does:** The demo: base and wrist together keep the block at the aim point. Move the block, hold it still until the display says CENTERED, repeat.

**Run:** **REQUIRED** for the demo. Press **q** in the window to stop.

**You should see:** TRACK while the block moves, CENTERED when it stops; the overlay (camera center, gripper z axis, x5/y5 arrows, offset in mm). Video and log saved.

**If it fails:** jerky: lower TRACK_A, check C.4's shaping. Hunting around the center: larger DEADBAND, or run C.3. Lost the block: slower moves of the block, or check the color range (A.6).

In [ ]:
# ===================== D.3 Track, 2 axes (REQUIRED for the demo) =====================
if SIM: world.target = np.array([440.0, 40.0, 270.0])
trk2 = new_tracker("2AXIS")
log2 = trk2.run(seconds=8 if SIM else None, show=SHOW, record=f"videos/{STAMP}_2axis.mp4")
trk2.save_log(f"logs/{STAMP}_2axis.csv"); SESSION["track_2axis"] = summary(log2)

## D.4 Compare the two detectors

**What it does:** Both detectors ran on every frame of the last run; this plots the tracking error and how far apart the two aim points were.

**Run:** **OPTIONAL** (for the report).

**You should see:** the mean and largest difference, two plots.

In [ ]:
# ===================== D.4 Compare detectors (OPTIONAL) =====================
L = [r for r in log2 if r["found"] and not np.isnan(r["alt_cx"])]
t = np.array([r["t"] for r in L]) - L[0]["t"]
d = np.hypot([r["cx"] - r["alt_cx"] for r in L], [r["cy"] - r["alt_cy"] for r in L])
print(f"{len(L)} frames: blob and edges aim points differ by mean {d.mean():.1f} px, max {d.max():.1f} px")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t, [r["e_px"] for r in L]); ax[0].axhline(trk2.cfg.deadband_in_px, color="g", ls=":"); ax[0].set_title("error, px"); ax[0].set_xlabel("s")
ax[1].plot(t, d); ax[1].set_title("blob vs edges aim point, px"); ax[1].set_xlabel("s")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()
SESSION["detectors"] = {"frames": len(L), "diff_mean_px": float(d.mean()), "diff_max_px": float(d.max())}

## D.5 Finish and save the session

**What it does:** Stops the arm (it keeps holding), saves a summary of this session's measurements (profile, Jacobian, slop, ringing, tracker settings, results) to `logs/<stamp>_session.json`, and releases the port and the camera.

**Run:** **REQUIRED** at the end.

**You should see:** the file name and 'port released'.

To let the arm go limp: **support it**, then run `lk.cmd("OFF *")` before this cell.

In [ ]:
# ===================== D.5 Finish (REQUIRED) =====================
lk.stop()
print(lk.get_state())
with open(f"logs/{STAMP}_session.json", "w") as f:
    json.dump(SESSION, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
print("saved", f"logs/{STAMP}_session.json")
cam.close(); lk.close(); cam = None; lk = None
print("port released: the Arduino Serial Monitor can open it again")